# Cuaderno de entrenamiento del modelo final
## Configuración del cuaderno


In [ ]:
#Instalación de paquetes
!pip install tf_keras tensorflow numpy matplotlib -q
!pip install coral-ordinal

import os

# Forzar uso de Keras 2 para evitar problemas de compatibilidad con STM32Cube.AI
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import shutil
import numpy as np
import tensorflow as tf
import tf_keras as keras
from tf_keras import layers, Model
import coral_ordinal as coral
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
)

# Verificar que estamos en Keras 2
assert int(keras.__version__.split('.')[0]) == 2, "No se está usando la versión de Keras2"
print("Usando Keras2")

from google.colab import drive
drive.mount('/content/drive')

import sys

SRC_PATH = "/content/drive/MyDrive/TFG/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

In [ ]:
# Macros

DATASET_ORI_PATH = '/content/drive/MyDrive/TFG/dt_ori'
DATASET_AUG_PATH = '/content/drive/MyDrive/TFG/dt_aug'
MODEL_PATH = '/content/drive/MyDrive/TFG/entreno_final'
RESULTADOS_PATH = f'{MODEL_PATH}/resultados_final'

os.makedirs(MODEL_PATH,   exist_ok=True)
os.makedirs(RESULTADOS_PATH, exist_ok=True)

IMAGE_SIZE = (480, 270)
INPUT_SHAPE = (224, 224, 3)

NUM_BATCHES = 32
NUM_EPOCH = 5

SEMAFOROS = np.array(['Benidorm', 'Daroca', 'Delicias', 'FrayLuis', 'Martires', 'MCerralbo', 'Pardinas'])
SEMAFOROS_TEST = np.array (['PJesusO'])

LABELS = {'0':'fluido', '1' : 'moderado', '2' : 'denso', '3' : 'saturado'}
NUM_CLASES = len(LABELS)


## Funciones auxiliares

In [ ]:
from preprocesado import ImageCropY, preprocesado
preprocesado = preprocesado(IMAGE_SIZE)
from dataset import build_dataset
from evaluacion import evaluar_modelo

## Definición del modelo: MobileNet

In [ ]:
def build_model() :
  #Definición del modelo
  base_model = keras.applications.MobileNet(
      input_shape= INPUT_SHAPE,
      alpha=0.50,
      include_top=False,
      weights='imagenet'
  )
  base_model.trainable = False

  inputs = keras.Input(shape=INPUT_SHAPE, name='input')
  x = base_model(inputs, training=False)
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dropout(0.4)(x)
  x = layers.Dense(128, activation='relu')(x)
  x = layers.Dropout(0.3)(x)
  outputs = coral.CoralOrdinal(num_classes=NUM_CLASES, name='output')(x)

  model = Model(inputs, outputs, name='MobileNet_a050_coral')

  # Compilación del modelo
  model.compile(
      optimizer=keras.optimizers.Adam(1e-3),
      loss=coral.OrdinalCrossEntropy (num_classes=NUM_CLASES),
      metrics=[
          coral.MeanAbsoluteErrorLabels(name='mae_labels')
      ]
  )

  return model


## Carga de datasets

In [ ]:
# Carga del dataset
print('Carga de los dataset: ')
ds_train = build_dataset(SEMAFOROS, DATASET_AUG_PATH, LABELS, IMAGE_SIZE, NUM_BATCHES, train=True)
ds_test = build_dataset(SEMAFOROS_TEST, DATASET_ORI_PATH, LABELS, IMAGE_SIZE, NUM_BATCHES, train=False)

# Preprocesado del dataset
ds_train = ds_train.map(lambda x,y : (preprocesado(x),y),
                        num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.map(lambda x,y : (preprocesado(x),y),
                      num_parallel_calls=tf.data.AUTOTUNE)



## Entrenamiento

In [ ]:
# Entrenamiento
print('Entrenamiento')
model = build_model()

history = model.fit(
    ds_train,
    epochs=NUM_EPOCH,
    verbose=1
)

model.save(f'{MODEL_PATH}/final_model.keras')

with open(f'{MODEL_PATH}/history.json', 'w') as f:
    json.dump(
        {k: [float(v) for v in vals] for k, vals in history.history.items()},
        f,
        indent=2
    )

### Evaluación del modelo

In [ ]:
custom_objects = {
    'ImageCropY': ImageCropY,
    'CoralOrdinal': coral.CoralOrdinal,
    'OrdinalCrossEntropy': coral.OrdinalCrossEntropy,
    'MeanAbsoluteErrorLabels': coral.MeanAbsoluteErrorLabels
}

# Test del modelo
print('Test')
model = keras.models.load_model(
    f'{MODEL_PATH}/final_model.keras',
    custom_objects
)
metricas = evaluar_modelo(model, ds_test, LABELS, RESULTADOS_PATH)

# Guardar resultados de test
with open(os.path.join(RESULTADOS_PATH, 'resultados_test_2.json'), 'w') as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)

## Fine-tuning

In [ ]:
NUM_EPOCH = 4

custom_objects = {
    'ImageCropY': ImageCropY,
    'CoralOrdinal': coral.CoralOrdinal,
    'OrdinalCrossEntropy': coral.OrdinalCrossEntropy,
    'MeanAbsoluteErrorLabels': coral.MeanAbsoluteErrorLabels
}

model = keras.models.load_model(
    f'{MODEL_PATH}/final_model.keras',
    custom_objects
)

base_model = model.get_layer('mobilenet_0.50_224')
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
  optimizer=keras.optimizers.Adam(learning_rate=1e-5),
  loss=coral.OrdinalCrossEntropy(num_classes=NUM_CLASES),
  metrics=[coral.MeanAbsoluteErrorLabels(name='mae_labels')]
)

In [ ]:
history = model.fit(
      ds_train,
      epochs=NUM_EPOCH,
      verbose=1
  )

model.save(f'{MODEL_PATH}/final_model_ft.keras')

with open(f'{MODEL_PATH}/history_ft.json', 'w') as f:
    json.dump(
        {k: [float(v) for v in vals] for k, vals in history.history.items()},
        f,
        indent=2
    )

### Evaluación del modelo

In [ ]:
RESULTADOS_PATH = f'{MODEL_PATH}/resultados_final_ft'
os.makedirs(RESULTADOS_PATH, exist_ok=True)

custom_objects = {
    'ImageCropY': ImageCropY,
    'CoralOrdinal': coral.CoralOrdinal,
    'OrdinalCrossEntropy': coral.OrdinalCrossEntropy,
    'MeanAbsoluteErrorLabels': coral.MeanAbsoluteErrorLabels
}

# Test del modelo
print('Test')
model = keras.models.load_model(
    f'{MODEL_PATH}/final_model_ft.keras',
    custom_objects
)
metricas = evaluar_modelo(model, ds_test, LABELS, RESULTADOS_PATH)

# Guardar resultados de test
with open(os.path.join(RESULTADOS_PATH, 'resultados_test_ft.json'), 'w') as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)